In [1]:
import pandas as pd
from top2vec import Top2Vec
import os
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import umap
import plotly.express as px

/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
stopwords=stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [3]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-stanza-lynch.feather')

In [4]:
data.shape

(223131, 6)

In [5]:
data.head(2)

,index,headline,newspaper_name,date,article,entities
3,5_1915-01-14_p1_sn88085445_0021110756A_1915011...,LAST CALL SOUNDS FOR\n\n\nCOMRADE WILCOXEN,"The Lynden tribune. [volume] (Lynden, Wash.) 1...",1915-01-14,"Charles Baxter WilcoXen, Civil\nwar veteran, p...","[[Charles Baxter WilcoXen, PERSON], [Friday, D..."
4,10_1915-08-21_p8_sn88051105_00513687928_191508...,"oh, That Lash!",The Harrison times.,1915-08-21,Tommy had watched the high-hatted\nand frock-c...,"[[Tommy, PERSON], [Tommy, PERSON], [Tommy, PER..."


In [6]:
data['article_stop'] = data['article'].str.lower().str.split().apply(lambda x: [word for word in x if word not in stopwords])


In [7]:
data['article_lemma'] = data['article_stop'].apply(lambda x: [WordNetLemmatizer().lemmatize(word) for word in x])

In [8]:
data['article_lemma_string']=data['article_lemma'].apply(lambda x: ' '.join(x))

In [84]:
Counter(data['article_lemma'].explode()).most_common(10)

[('one', 173474),
 ('|', 132698),
 ('would', 115037),
 ('mrs.', 112925),
 ('new', 110770),
 ('state', 109872),
 ('1t', 98776),
 ('two', 96909),
 ('whip', 95542),
 ('10', 91046)]

In [58]:
model=Top2Vec(documents=data['article_lemma_string'].tolist(), speed="learn", workers=4, embedding_model='distiluse-base-multilingual-cased')

2025-11-13 14:45:48,179 - top2vec - INFO - Pre-processing documents for training
/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'

2025-11-13 14:46:29,061 - top2vec - INFO - Downloading distiluse-base-multilingual-cased model
2025-11-13 14:46:31,968 - top2vec - INFO - Creating joint document/word embedding
2025-11-13 14:58:22,325 - top2vec - INFO - Creating lower dimension embedding of documents
2025-11-13 14:58:54,084 - top2vec - INFO - Finding dense areas of documents
/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

202

In [59]:
topic_sizes, topic_nums = model.get_topic_sizes()

In [ ]:
print(len(topic_sizes), len(topic_nums)) #955 topics with distiluse-base-multilingual-cased

955 955


In [61]:
id_dic={}
topic_id={}
for element in zip(topic_nums, topic_sizes):
    documents, document_scores, document_ids = model.search_documents_by_topic(topic_num=element[0], num_docs=element[1])
    for score, id in zip(document_scores, document_ids):
        id_dic[id]=score
        topic_id[id]=element[0]

In [62]:
topic_words, word_scores, topic_scores = model.get_topics(len(topic_sizes))

In [63]:
df_words=pd.DataFrame(topic_words).transpose()
df_words.columns=topic_nums
df_words.columns = df_words.columns.astype(str)

In [83]:
df_words[['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']].iloc[:10]

,0,1,2,3,4,5,6,7,8,9
0,senate,fight,racing,baseball,madame,negro,theatrical,shouted,deutsche,policeman
1,filibuster,sparring,racetrack,inning,miss,blacks,theatre,screamed,german,policia
2,senat,brawl,horsewhipped,pitcher,fortnight,negroes,dramatic,grunted,germany,policemen
3,legislature,boxing,marathon,innings,domingo,blackman,shakespeare,yelled,deutschland,police
4,legislatures,fights,horsewhipping,pitchers,wednesday,blackwell,drama,whispered,germanic,arrests
5,congressional,fought,raced,umpires,mckinney,niblack,actor,scream,nazis,sheriff
6,congressman,wrestling,racer,batting,misses,blackness,actress,yell,nazism,arrest
7,parliamentary,fightin,horses,batsman,weekend,nigger,ator,roared,guerra,arresting
8,senatorial,undefeated,horseback,umpire,poned,niggers,acting,shout,war,cops
9,senator,wrestle,competed,pitching,weekday,policia,hollywood,screams,guerre,sheriffs


In [65]:
data['topic_id']=data.index.map(topic_id) #map topic id to each document
data['topic_score']=data.index.map(id_dic) #map topic score to each document

In [67]:
Counter(data['topic_id']).most_common(20)

[(0.0, 3426),
 (1.0, 3316),
 (2.0, 1847),
 (3.0, 1758),
 (6.0, 1581),
 (4.0, 1580),
 (5.0, 1579),
 (7.0, 1512),
 (8.0, 1422),
 (9.0, 1361),
 (10.0, 1316),
 (11.0, 1300),
 (12.0, 1212),
 (13.0, 1156),
 (14.0, 1124),
 (17.0, 968),
 (15.0, 966),
 (16.0, 948),
 (18.0, 903),
 (19.0, 869)]

In [69]:
df_words.to_csv('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-stanza-lynch-topics-words.csv', index=False)

In [70]:
data['topic_id'].isna().sum()

np.int64(93843)

In [71]:
data[data['topic_id'].isna()].shape

(93843, 12)

In [72]:
data[['article', 'topic_id', 'topic_score']].head(10)

,article,topic_id,topic_score
3,"Charles Baxter WilcoXen, Civil\nwar veteran, p...",80.0,0.744918
4,Tommy had watched the high-hatted\nand frock-c...,23.0,0.590569
5,"KANSAS CITY, Mo-, Sept. S-The lead\ners of the...",504.0,0.631608
6,Washington. Jan. %. The Senate tug\nOf war ove...,465.0,0.522632
8,"The advance sale of tickets fo~\n""The Clansman...",59.0,0.686477
10,doing a service to some one he could\nnot see....,42.0,0.622413
11,+Parsons-souders Co.\n.Sturm's Millinery.\nCIe...,354.0,0.623641
12,Crush a large cupful Of strawberries\nand mix ...,251.0,0.645561
13,board reported successful year\nfinancially an...,832.0,0.975620
14,Don't whip the horse if he is afraid.\nTalk ge...,288.0,0.467901


In [73]:
data['vector']=model.document_vectors.tolist()

In [74]:
data.to_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-top2vec.feather')

In [2]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-top2vec.feather')

In [3]:
data['topic_id'].nunique()

955

In [4]:
notnull_data=data[data['topic_id'].notna()]

In [5]:
embedding = umap.UMAP().fit_transform(notnull_data['vector'].tolist()) #reduce dimensionality of vector representation

In [6]:
clusterable_embedding = umap.UMAP(
    n_neighbors=30,
    min_dist=0.0,
    n_components=3,
    random_state=42,
).fit_transform(embedding.data)

/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: R

In [ ]:
fig = px.scatter_3d(
    notnull_data, 
    x=clusterable_embedding[:, 0], 
    y=clusterable_embedding[:, 1], 
    z=clusterable_embedding[:, 2], 
    color='topic_id',  # Specify the column for color
    hover_data=['topic_id']
)

fig.update_layout(
    autosize=False,
    width=1000,
    height=1000,
    title={"text": "The Vector Representation of KKK Articles with UMAP Dimension Reduction",
            "x": 0.5,
            "xanchor": "center"},
    scene=dict(
        aspectmode="manual",
        aspectratio=dict(x=1.3, y=1.3, z=1.3)  # ⬅ bigger plotting box
    ),
    scene_camera=dict(
        eye=dict(x=2, y=2, z=2)  # zoom out view
    )
)

fig.update_traces(marker=dict(size=3))

# fig.show()
fig.write_html("/Volumes/T7/chroniclingamerica/american-stories/interactive-plot-3d.html")
fig.write_image("/Volumes/T7/chroniclingamerica/american-stories/umap-3d.png")
#when hover over the dots, the topic id and doi will show up as well as x and y coordinates